### 프롬프트 템플릿  Agent tool rag에서 표준
- role:LLM이 어떤 역활을 할지 정함
- instruction:답변 규칙과 형식을 정리
- context:데이터베이스 검색결과처럼 답변에 참고할 실제 정보
- query:실제 질문


In [1]:
# 사용자 질문 -> LLM개입해서 사용자 의도를 파악해서
# 적절한 tool 호출 -> 라우터( Fast API )
# 수집한 정보를 -> LLM 전달
# 최종답변은 LLM

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    raise EnvironmentError('openai api key .....')

class OpenAILLM:
    def __init__(self,model:str = 'gpt-5.4-nano'):
        self.client = OpenAI(api_key=api_key)
        self.model = model
    def generate(self, prompt:str) -> str:
        response = self.client.chat.completions.create(
            model = self.model,
            messages =[
                {"role":"system", "content":"You are an ecomerce recommendation assistant, Return only valid JSON"},
                {"role":"user", "content":prompt}
            ],
            temperature=0,
            response_format={'type':'json_object'}
        )
        return response.choices[0].message.content
llm = OpenAILLM()
llm.model  

'gpt-5.4-nano'

In [3]:
# 데이터
do_search_results = [
    {"id":"p1", "name":"고성능 노트북","category":"가전"},
    {"id":"p2", "name":"사무용 랩탑","category":"가전"},
    {"id":"p3", "name":"미러리스 카메라","category":"가전"},
]
context_string = ""
for item in do_search_results:
    context_string += f"- 상품명:{item['name']} | 카테고리:{item['category']}"

In [4]:
# 프롬프트 템플릿 - 고정된 구조
from  textwrap import dedent  #  들여쓰기 indent를 제거
user_query = '코딩할 때 쓸만한 노트북 하나 추천해줘, 믿을만한 제품으로'
raw_prompt = dedent(f'''
                    당신은 이커머스 전문 AI 추천 어시스턴트입니다.
                    아래 컨텍스트를 참고하여 사용자의 질문에 답하세요
                    반드시 json 형식으로만 답하세요

                    [컨텍스트]
                    {context_string}

                    [질문]
                    {user_query}

                    답변은 반드시 아래 형식의 json으로만 출력하세요
                    {
                        {
                            "recommended_product" : "상품명",
                            "reason":"추천사유"                            
                        }
                    }
                    ''')
print(raw_prompt)


당신은 이커머스 전문 AI 추천 어시스턴트입니다.
아래 컨텍스트를 참고하여 사용자의 질문에 답하세요
반드시 json 형식으로만 답하세요

[컨텍스트]
- 상품명:고성능 노트북 | 카테고리:가전- 상품명:사무용 랩탑 | 카테고리:가전- 상품명:미러리스 카메라 | 카테고리:가전

[질문]
코딩할 때 쓸만한 노트북 하나 추천해줘, 믿을만한 제품으로

답변은 반드시 아래 형식의 json으로만 출력하세요
{'recommended_product': '상품명', 'reason': '추천사유'}



### LLM 응답

In [5]:
import json
response = llm.generate(raw_prompt)
data = json.loads(response)
print(json.dumps(data, indent=2, ensure_ascii=False))

{
  "recommended_product": "고성능 노트북",
  "reason": "코딩 작업에는 CPU 성능과 메모리/저장공간이 중요해서, 고성능 노트북이 개발 환경(IDE, 로컬 서버, 여러 탭/프로세스) 구동에 더 안정적입니다. 또한 가전 카테고리 내에서 성능 중심 제품이라 믿고 사용하기 좋습니다."
}


### 라우팅
- 들어온 질문을 보고 어느 경로로 보낼지 결정하는 단계

In [6]:
def route_qeustion(question:str)->str:
    lower_question = question.lower()
    if any(keyword in lower_question for keyword in ['뉴스','기사','검색','찾아줘','최신','오늘']):
        return "news_tool"
    elif any(keyword in lower_question for keyword in ['계산','더하기','곱하기','합계','몇','얼마']):
        return "calculator_tool"
    elif any(keyword in lower_question for keyword in ['기억','기록','선호','메모','이전']):
        return "memory_tool"
    elif any(keyword in lower_question for keyword in ['추천','골라','비교']):
        return "llm_recomendation"
    else:
        return "general_llm"
sample_questions = [
    "3개 상품을 2개씩 주문하면 총 몇 개인가?",
    "나는 코딩용 노트북을 좋아한다는 점을 기억해줘.",
    "AI 에이전트 뉴스 최신 기사 3개 찾아줘.",
    "코딩할 때 쓸만한 노트북 추천해줘.",
]
for question in sample_questions:
    print(f'질문 : {question} | 라우트 : {route_qeustion(question)}')    

질문 : 3개 상품을 2개씩 주문하면 총 몇 개인가? | 라우트 : calculator_tool
질문 : 나는 코딩용 노트북을 좋아한다는 점을 기억해줘. | 라우트 : memory_tool
질문 : AI 에이전트 뉴스 최신 기사 3개 찾아줘. | 라우트 : news_tool
질문 : 코딩할 때 쓸만한 노트북 추천해줘. | 라우트 : llm_recomendation


### tool 활용

In [8]:
import json
import os
import re
from urllib.parse import quote
from urllib.request import Request,urlopen


def calcualtor_tool(text:str)->float:
    allowed_chars = set("0123456789+-*/().")
    if not set(text) <= allowed_chars: # text의 문자중에 허용되지 않은 문자가 있다면
        raise ValueError('허용되지 않은 문자가 포함되어 있습니다.')
    return eval(text)

In [9]:
# 네이버 검색 API 예제 - 블로그 검색
import os
import re
import sys
import json
import html
import urllib.request
from datetime import datetime
from dotenv import load_dotenv
load_dotenv(override=True)

def _format_date(pubdate):
    return datetime.strptime(pubdate, "%a, %d %b %Y %H:%M:%S %z").strftime("%Y-%m-%d")

def _format_str(text):
    return html.unescape(re.sub(r'<[^>]+>',"",text))

client_id = os.getenv('NAVER_CLIENT_ID')
client_secret = os.getenv('NAVER_CLIENT_SECETET')

items = []
def search_naver_news(query:str, display:int=3)->list[dict]:
    encText = urllib.parse.quote(str)
    encText+= f'&display={display}&sort=date'
    url = "https://openapi.naver.com/v1/search/news?query=" + encText # JSON 결과    
    request = urllib.request.Request(url)
    request.add_header("X-Naver-Client-Id",client_id)
    request.add_header("X-Naver-Client-Secret",client_secret)


    response = urllib.request.urlopen(request)
    rescode = response.getcode()
    if(rescode==200):
        response_body = response.read().decode('utf-8')
        result = json.loads(response_body)

        for row in result.get('items'):
            items.append({
                'title':_format_str(row.get('title')),
                'content':_format_str(row.get('description')),                
                'date':_format_date(row.get('pubDate')),
                'link':row.get('link')
            })         
    return items